# ETL: Entregable 1

In [1]:
from dotenv import dotenv_values
from sqlalchemy import create_engine, text
from datetime import datetime, timedelta
import pandas as pd
from spotipy import Spotify
from spotipy.oauth2 import SpotifyOAuth
from helpers import check_if_valid_data, psql_insert_copy

In [2]:
config = dotenv_values(".env")

In [3]:
# Spotify settings
CLIENT_ID = config["CLIENT_ID"]
CLIENT_SECRET = config["CLIENT_SECRET"]
SPOTIPY_REDIRECT_URI = config["SPOTIPY_REDIRECT_URI"]
SCOPE = config["SCOPE"]

In [4]:
# DB settings
TABLENAME = config["TABLENAME"]
DB_USERNAME = config["DB_USERNAME"]
DB_PASSWORD = config["DB_PASSWORD"]
DB_HOST = config["DB_HOST"]
DB_PORT = config["DB_PORT"]
DB_NAME = config["DB_NAME"]
DB_SCHEMA = config["DB_SCHEMA"]

In [5]:
# Conexión a Spotify
sp = Spotify(auth_manager=SpotifyOAuth(client_id=CLIENT_ID,
                                               client_secret=CLIENT_SECRET,
                                               redirect_uri=SPOTIPY_REDIRECT_URI,
                                               scope=SCOPE))

# Conviertir una timestamp de unix en milisegundos
today = datetime.now()
yesterday = today - timedelta(days=1)
yesterday = yesterday.replace(hour=0, minute=0, second=0, microsecond=0)
yesterday_unix_timestamp = int(yesterday.timestamp()) * 1000
limit = 50    

# Retorna un diccionario con las canciones escuchadas
raw_data = sp.current_user_recently_played(limit=limit, after=yesterday_unix_timestamp)

In [6]:
raw_data.keys()

dict_keys(['items', 'next', 'cursors', 'limit', 'href'])

In [7]:
song_names = []
artist_names = []
played_at_list = []
timestamps = []

In [8]:
# Guardamos la data en listas
for song in raw_data["items"]:
    song_names.append(song["track"]["name"])
    artist_names.append(song["track"]["artists"][0]["name"])
    played_at_list.append(song["played_at"]),
    timestamps.append(song["played_at"][0:10])

In [9]:
# Creamos un diccionario de listas
song_dict = {
    "song_name" : song_names,
    "artist_name": artist_names,
    "played_at" : played_at_list,
    "timestamp" : timestamps
}

In [10]:
# Crear el DataFrame
song_df = pd.DataFrame(song_dict, columns = ["song_name",
                                             "artist_name",
                                             "played_at",
                                             "timestamp"])

In [19]:
# Transform

# Eliminar fechas diferentes a las que queremos
clean_song_df = song_df[pd.to_datetime(song_df["played_at"]).dt.date == yesterday.date()]

# Seteamos los tipo de datos para las columnas
clean_song_df["song_name"] = clean_song_df["song_name"].astype(str)

clean_song_df["artist_name"] = clean_song_df["artist_name"].astype(str)

clean_song_df["played_at"] = pd.to_datetime(clean_song_df["played_at"].astype(str),
                                            format="%Y-%m-%d %H:%M:%S.%f",
                                            errors="ignore")

clean_song_df["timestamp"] = pd.to_datetime(clean_song_df["timestamp"])

In [ ]:
clean_song_df.groupby(["artist_name"])["song_name"].count()

In [ ]:
# Creamos una nueva columna
clean_song_df["number_of_songs_by_artist"] = song_df.groupby(["artist_name"])["song_name"].transform("count")

In [27]:
clean_song_df

,song_name,artist_name,played_at,timestamp


# Conexión a PostgreSQL en local

In [ ]:
# Validadcion de datos
if check_if_valid_data(clean_song_df):
    print("Data valid, proceed to Load stage...")

    # Load

    # Crear objeto engine para conecatarse a la base de datos
    conn_string = f"postgresql+psycopg2://{DB_USERNAME}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
    engine = create_engine(conn_string)

    # Utilizar el bloque with sirve para no preocuparnos de cerrar la conn a la db
    # el bloque lo hace solo.
    with engine.connect() as conn:
        # Ejecuta la consulta
        sql_query = text("""
        CREATE TABLE IF NOT EXISTS coder.my_tracks_history(
            id_my_tracks_history SERIAL NOT NULL,
            song_name VARCHAR(200) NOT NULL,
            artist_name VARCHAR(200) NOT NULL,
            played_at TIMESTAMP NOT NULL,
            timestamp DATE NOT NULL,

            CONSTRAINT pk_my_tracks_history PRIMARY KEY (played_at)
        );
        """)
        conn.execute(sql_query)

        rows_imported = 0
        print(f"Importing rows {rows_imported} to {rows_imported + clean_song_df.shape[0]}... for table {TABLENAME}")

        # Cargamos los nuevos datos a la tabla
        clean_song_df.to_sql(
            name="my_tracks_history",
            con=engine,
            schema="coder",
            if_exists="append",
            index=False,
            chunksize=1000,
            method=psql_insert_copy
        )

        rows_imported += clean_song_df.shape[0]
        print(f"Importing rows {rows_imported} to {clean_song_df.shape[0]}... for table {TABLENAME}")
        print(f"Data imported successful")

        # close conn
        engine.dispose()
else:
    print("No songs downloaded. Finishing execution")

# Conexión a AWS Redshift

In [40]:
# AWS Redshift settings
REDSHIFT_HOST = config["AWS_HOST"]
REDSHIFT_PORT = int(config["AWS_PORT"])
REDSHIFT_DATABASE = config["AWS_DB_NAME"]
REDSHIFT_SCHEMA = config["AWS_DB_SCHEMA"]
REDSHIFT_USERNAME = config["AWS_USER"]
REDSHIFT_PASS = config["AWS_PASSWORD"]

In [44]:
# Validadcion de datos
if check_if_valid_data(clean_song_df):
    print("Data valid, proceed to Load stage...")
    
    # Load
    
    # Crear objeto engine para conecatarse a la base de datos
    engine = create_engine(f"redshift+psycopg2://{REDSHIFT_USERNAME}:{REDSHIFT_PASS}@{REDSHIFT_HOST}:{REDSHIFT_PORT}/{REDSHIFT_DATABASE}")
    conn = engine.connect()

    with engine.connect() as conn:
        # Ejecuta la consulta
        sql_query = """
        CREATE TABLE IF NOT EXISTS lautaropoletto_coderhouse.my_tracks_history(
            id_my_tracks_history bigint identity(0, 1) NOT NULL,
            song_name VARCHAR(200) NOT NULL,
            artist_name VARCHAR(200) NOT NULL,
            played_at TIMESTAMP NOT NULL,
            timestamp DATE NOT NULL,
            primary key(played_at))
            distkey(id_my_tracks_history)
            compound sortkey(timestamp, artist_name);
        """
        conn.execute(sql_query)
        
        rows_imported = 0
        table_name = "my_tracks_history"

        # Cargamos los nuevos datos a la tabla
        print(f"Importing rows {rows_imported} to {rows_imported + clean_song_df.shape[0]}... for table {table_name}")

        clean_song_df.to_sql(table_name,
                    con=engine,
                    schema=REDSHIFT_SCHEMA,
                    if_exists="append",
                    index=False,
                    chunksize=1000,
                    method=psql_insert_copy)

        rows_imported += clean_song_df.shape[0]
        print(f"Importing rows {rows_imported} to {clean_song_df.shape[0]}... for table {table_name}")
        print(f"Data imported successful")

        conn.close()
else:
    print("No songs downloaded. Finishing execution")

Importing rows 0 to 22... for table my_tracks_history
Importing rows 22 to 22... for table my_tracks_history
Data imported successful
